### Simplified Self-attention

In [1]:
import sys
from pathlib import Path

sys.path.append(str(Path.cwd().parent))

In [2]:
import torch
import torch.nn.functional as F

In [3]:
torch.manual_seed(42)

In [4]:
# 6 words, each a 3-dim embedding
X = torch.rand(6, 3)
X

tensor([[0.8823, 0.9150, 0.3829],
        [0.9593, 0.3904, 0.6009],
        [0.2566, 0.7936, 0.9408],
        [0.1332, 0.9346, 0.5936],
        [0.8694, 0.5677, 0.7411],
        [0.4294, 0.8854, 0.5739]])

In [5]:
# Step 1: attention SCORES — every word dotted with every word.
scores = X @ X.T
scores

tensor([[1.7622, 1.4337, 1.3127, 1.1999, 1.5702, 1.4088],
        [1.4337, 1.4338, 1.1213, 0.8494, 1.5010, 1.1025],
        [1.3127, 1.1213, 1.5807, 1.3343, 1.3708, 1.3528],
        [1.1999, 0.8494, 1.3343, 1.2435, 1.0863, 1.2254],
        [1.5702, 1.5010, 1.3708, 1.0863, 1.6274, 1.3013],
        [1.4088, 1.1025, 1.3528, 1.2254, 1.3013, 1.2978]])

In [6]:
# Step 2: attention WEIGHTS — softmax each row so it sums to 1.
weights = F.softmax(scores, dim=1)
weights

tensor([[0.2244, 0.1616, 0.1432, 0.1279, 0.1852, 0.1576],
        [0.1970, 0.1970, 0.1441, 0.1098, 0.2107, 0.1414],
        [0.1599, 0.1320, 0.2090, 0.1633, 0.1694, 0.1664],
        [0.1721, 0.1212, 0.1968, 0.1798, 0.1536, 0.1765],
        [0.1926, 0.1797, 0.1578, 0.1187, 0.2039, 0.1472],
        [0.1884, 0.1387, 0.1782, 0.1569, 0.1692, 0.1686]])

In [7]:
for i, row in enumerate(weights):
    print(f"sum of row {i+1} : {sum(row)}")

sum of row 1 : 0.9999999403953552
sum of row 2 : 1.0
sum of row 3 : 0.9999999403953552
sum of row 4 : 1.0
sum of row 5 : 1.0
sum of row 6 : 0.9999999403953552


In [8]:
# Step 3: CONTEXT vectors — each output is a weighted sum of all input words.
context = weights @ X
context

tensor([[0.6355, 0.7464, 0.6214],
        [0.6583, 0.7190, 0.6319],
        [0.5618, 0.7598, 0.6551],
        [0.5519, 0.7725, 0.6457],
        [0.6392, 0.7287, 0.6363],
        [0.5854, 0.7599, 0.6384]])

### Self Attention with Trainable Parameters

In [9]:
# Input - 6 words, each a 3-dim embedding
X = torch.rand(6, 3)
X

tensor([[0.2666, 0.6274, 0.2696],
        [0.4414, 0.2969, 0.8317],
        [0.1053, 0.2695, 0.3588],
        [0.1994, 0.5472, 0.0062],
        [0.9516, 0.0753, 0.8860],
        [0.5832, 0.3376, 0.8090]])

In [10]:
# Step 1 - Define the query, key and value metrics
w_q = torch.rand(3, 3)
w_k = torch.rand(3, 3)
w_v = torch.rand(3, 3)

print(f"dimension of w_q: {w_q.size()}")

dimension of w_q: torch.Size([3, 3])


In [11]:
# Step 2 - Produce the Q, K and V
Q = X @ w_q
K = X @ w_k
V = X @ w_v

print(f"dimension of Q is : {Q.size()}")

dimension of Q is : torch.Size([6, 3])


In [12]:
# Step 3 - Calculate the attention scores
attention_scores = Q @ K.T
attention_scores

tensor([[0.9182, 1.3881, 0.5807, 0.5746, 1.9166, 1.5657],
        [1.5077, 2.2910, 0.9625, 0.9368, 3.1390, 2.5761],
        [0.6564, 0.9987, 0.4189, 0.4077, 1.3708, 1.1235],
        [0.5155, 0.7740, 0.3227, 0.3252, 1.0766, 0.8759],
        [1.9051, 2.8831, 1.2124, 1.1874, 3.9509, 3.2437],
        [1.6370, 2.4828, 1.0429, 1.0190, 3.4051, 2.7933]])

In [13]:
# Step 4 - Scale the attention_scores
scaled_attention_scores = attention_scores  / (K.size()[-1] ** 0.5)
scaled_attention_scores

tensor([[0.5301, 0.8014, 0.3353, 0.3318, 1.1066, 0.9040],
        [0.8705, 1.3227, 0.5557, 0.5409, 1.8123, 1.4873],
        [0.3790, 0.5766, 0.2419, 0.2354, 0.7914, 0.6486],
        [0.2976, 0.4469, 0.1863, 0.1878, 0.6216, 0.5057],
        [1.0999, 1.6646, 0.7000, 0.6855, 2.2810, 1.8727],
        [0.9451, 1.4334, 0.6021, 0.5883, 1.9659, 1.6127]])

In [14]:
# Step 5 - Calculate the attention weights
attention_weights = F.softmax(scaled_attention_scores, dim = -1)
attention_weights

tensor([[0.1391, 0.1825, 0.1145, 0.1141, 0.2476, 0.2022],
        [0.1185, 0.1863, 0.0865, 0.0852, 0.3039, 0.2196],
        [0.1476, 0.1798, 0.1287, 0.1278, 0.2229, 0.1932],
        [0.1523, 0.1768, 0.1363, 0.1365, 0.2106, 0.1875],
        [0.1051, 0.1849, 0.0705, 0.0695, 0.3424, 0.2277],
        [0.1141, 0.1859, 0.0810, 0.0799, 0.3167, 0.2224]])

In [15]:
# Step 6 - Calculate the context vector
context_vectors = attention_weights @ V
context_vectors

tensor([[0.8102, 0.7419, 0.7812],
        [0.8582, 0.7640, 0.8449],
        [0.7878, 0.7313, 0.7517],
        [0.7753, 0.7256, 0.7352],
        [0.8870, 0.7772, 0.8835],
        [0.8679, 0.7685, 0.8579]])

In [16]:
from src.attention import SelfAttention

X = torch.rand(6, 3)
attn = SelfAttention(3, 3)
context = attn(X)
context.shape

torch.Size([6, 3])

### Causal Attention

In [17]:
# Input - 6 words, each a 3-dim embedding
X = torch.rand(6, 3)
X

tensor([[0.0624, 0.1816, 0.9998],
        [0.5944, 0.6541, 0.0337],
        [0.1716, 0.3336, 0.5782],
        [0.0600, 0.2846, 0.2007],
        [0.5014, 0.3139, 0.4654],
        [0.1612, 0.1568, 0.2083]])

In [18]:
# Step 1 - Define the query, key and value metrics
w_q = torch.rand(3, 3)
w_k = torch.rand(3, 3)
w_v = torch.rand(3, 3)

print(f"dimension of w_q: {w_q.size()}")

dimension of w_q: torch.Size([3, 3])


In [19]:
# Step 2 - Produce the Q, K and V
Q = X @ w_q
K = X @ w_k
V = X @ w_v

print(f"dimension of Q is : {Q.size()}")
print(f"dimension of K is : {K.size()}")
print(f"dimension of V is : {V.size()}")

dimension of Q is : torch.Size([6, 3])
dimension of K is : torch.Size([6, 3])
dimension of V is : torch.Size([6, 3])


In [20]:
# Step 3 - Calculate the attention scores
attention_scores = Q @ K.T
attention_scores

tensor([[1.1456, 0.6939, 0.8833, 0.4556, 0.8022, 0.3645],
        [1.0226, 0.9688, 0.9059, 0.5264, 0.8604, 0.3926],
        [0.9793, 0.6864, 0.7863, 0.4210, 0.7248, 0.3297],
        [0.5328, 0.3832, 0.4308, 0.2317, 0.3995, 0.1814],
        [0.9899, 0.8306, 0.8415, 0.4742, 0.7866, 0.3592],
        [0.4356, 0.3441, 0.3629, 0.2009, 0.3379, 0.1540]])

In [21]:
# Step 4 - Scale the attention_scores
scaled_attention_scores = attention_scores  / (K.size()[-1] ** 0.5)
scaled_attention_scores

tensor([[0.6614, 0.4006, 0.5100, 0.2631, 0.4632, 0.2105],
        [0.5904, 0.5593, 0.5230, 0.3039, 0.4968, 0.2267],
        [0.5654, 0.3963, 0.4540, 0.2431, 0.4185, 0.1903],
        [0.3076, 0.2213, 0.2487, 0.1338, 0.2306, 0.1047],
        [0.5715, 0.4795, 0.4858, 0.2738, 0.4541, 0.2074],
        [0.2515, 0.1987, 0.2095, 0.1160, 0.1951, 0.0889]])

In [22]:
# Step 5 - Create the mask
mask = torch.triu(torch.ones(X.shape[0], X.shape[0]), diagonal=1)
mask

tensor([[0., 1., 1., 1., 1., 1.],
        [0., 0., 1., 1., 1., 1.],
        [0., 0., 0., 1., 1., 1.],
        [0., 0., 0., 0., 1., 1.],
        [0., 0., 0., 0., 0., 1.],
        [0., 0., 0., 0., 0., 0.]])

In [23]:
# Step 6 - Mask the attention scores
masked_attn_scores = scaled_attention_scores.masked_fill(mask.bool(), float('-inf'))
masked_attn_scores

tensor([[0.6614,   -inf,   -inf,   -inf,   -inf,   -inf],
        [0.5904, 0.5593,   -inf,   -inf,   -inf,   -inf],
        [0.5654, 0.3963, 0.4540,   -inf,   -inf,   -inf],
        [0.3076, 0.2213, 0.2487, 0.1338,   -inf,   -inf],
        [0.5715, 0.4795, 0.4858, 0.2738, 0.4541,   -inf],
        [0.2515, 0.1987, 0.2095, 0.1160, 0.1951, 0.0889]])

In [24]:
# Step 7 - Calculate the attention weights
attention_weights = F.softmax(masked_attn_scores, dim = -1)
attention_weights

tensor([[1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.5078, 0.4922, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.3651, 0.3083, 0.3266, 0.0000, 0.0000, 0.0000],
        [0.2702, 0.2479, 0.2548, 0.2271, 0.0000, 0.0000],
        [0.2241, 0.2044, 0.2057, 0.1664, 0.1993, 0.0000],
        [0.1793, 0.1701, 0.1720, 0.1566, 0.1695, 0.1524]])

In [25]:
# Step 8 - Calculate the context vector
context_vectors = attention_weights @ V
context_vectors

tensor([[0.8157, 0.4961, 0.5112],
        [0.8365, 0.7411, 0.5221],
        [0.8021, 0.6793, 0.4889],
        [0.7127, 0.6130, 0.4152],
        [0.7330, 0.6426, 0.4613],
        [0.6666, 0.5877, 0.4210]])

In [26]:
from src.attention import SelfAttention

X = torch.rand(6, 3)
attn = SelfAttention(3, 3, causal=False)
context = attn(X)
context.shape

torch.Size([6, 3])

In [27]:
attn = SelfAttention(3, 3, causal=True)
X = torch.rand(6, 3)

# reproduce the internals to see the weights
Q = attn.w_q(X)
K = attn.w_k(X)
scores = Q @ K.transpose(-2, -1) / (K.shape[-1] ** 0.5)

seq_len = scores.shape[-1]
mask = torch.triu(torch.ones(seq_len, seq_len), diagonal=1).bool()
scores = scores.masked_fill(mask, float('-inf'))

weights = torch.softmax(scores, dim=-1)
print(weights)
print("row sums:", weights.sum(dim=-1))

tensor([[1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.5013, 0.4987, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.3534, 0.3222, 0.3244, 0.0000, 0.0000, 0.0000],
        [0.2549, 0.2458, 0.2413, 0.2580, 0.0000, 0.0000],
        [0.2000, 0.2002, 0.1992, 0.2003, 0.2004, 0.0000],
        [0.1682, 0.1647, 0.1638, 0.1693, 0.1657, 0.1683]],
       grad_fn=<SoftmaxBackward0>)
row sums: tensor([1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000],
       grad_fn=<SumBackward1>)


### Multi-Head Attention

In [28]:
from src.attention import MultiHeadAttention

X = torch.rand(2, 6, 12)              # B=2, T=6, dim_in=12
mha = MultiHeadAttention(12, 12, num_heads=3)
out = mha(X)
print(out.shape) 

torch.Size([2, 6, 12])


In [29]:
mha = MultiHeadAttention(12, 12, num_heads=3, causal=True)
out = mha(torch.rand(2, 6, 12))
print(out.shape)   # (2, 6, 12)

torch.Size([2, 6, 12])


In [30]:
mha1 = MultiHeadAttention(12, 12, num_heads=1)
print(mha1(torch.rand(2, 6, 12)).shape)   # (2, 6, 12)

torch.Size([2, 6, 12])
